# 🗺️ Thu thập dữ liệu GreenMap từ OpenStreetMap

Notebook này hướng dẫn cách **tự động thu thập dữ liệu** địa lý từ OpenStreetMap cho dự án GreenMap.

## 📋 Các bước thực hiện:
1. Cài đặt thư viện cần thiết
2. Định nghĩa hàm gọi Overpass API
3. Cấu hình các query
4. Tự động thu thập dữ liệu
5. Hiển thị thống kê
6. Visualize trên bản đồ

## 🗺️ Khu vực thu thập:
- **Bounding Box:** (20.57, 105.28, 21.39, 106.02)
- **Khu vực:** Hà Nội và vùng lân cận

---
## 1. 📦 Cài đặt thư viện

In [ ]:
# Cài đặt các thư viện cần thiết
# Chỉ chạy cell này nếu chưa cài đặt
!pip install requests pandas folium --quiet

print("✅ Đã cài đặt các thư viện cần thiết!")

In [ ]:
# Import các thư viện
import requests
import json
import os
import time
from datetime import datetime

import pandas as pd

# Folium để tạo bản đồ tương tác (optional)
try:
    import folium
    from folium.plugins import MarkerCluster
    FOLIUM_AVAILABLE = True
    print("✅ Đã import folium thành công!")
except ImportError:
    FOLIUM_AVAILABLE = False
    print("⚠️ Không tìm thấy folium. Chức năng visualize sẽ bị bỏ qua.")

print("✅ Đã import tất cả thư viện cần thiết!")

---
## 2. 🔧 Định nghĩa hàm query Overpass API

In [ ]:
# URL của Overpass API
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Thư mục lưu dữ liệu
DATA_DIR = "Data"

def query_overpass(query, timeout=60):
    """
    Gọi Overpass API và trả về dữ liệu JSON.
    
    Args:
        query (str): Chuỗi query Overpass QL
        timeout (int): Thời gian chờ tối đa (giây)
    
    Returns:
        dict: Dữ liệu JSON từ API hoặc None nếu có lỗi
    """
    try:
        print(f"  ⏳ Đang gửi request đến Overpass API...")
        
        # Gửi POST request với query
        response = requests.post(
            OVERPASS_URL,
            data={"data": query},
            timeout=timeout
        )
        
        # Kiểm tra response status
        response.raise_for_status()
        
        # Parse JSON response
        data = response.json()
        
        print(f"  ✅ Nhận được {len(data.get('elements', []))} phần tử")
        return data
        
    except requests.exceptions.Timeout:
        print(f"  ❌ Lỗi: Request timeout sau {timeout} giây")
        print(f"      Thử tăng timeout hoặc thu nhỏ bounding box")
        return None
        
    except requests.exceptions.HTTPError as e:
        print(f"  ❌ Lỗi HTTP: {e}")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"  ❌ Lỗi kết nối: {e}")
        return None
        
    except json.JSONDecodeError as e:
        print(f"  ❌ Lỗi parse JSON: {e}")
        return None


def osm_to_geojson(osm_data):
    """
    Chuyển đổi dữ liệu OSM sang định dạng GeoJSON.
    
    Args:
        osm_data (dict): Dữ liệu từ Overpass API
    
    Returns:
        dict: Dữ liệu định dạng GeoJSON
    """
    features = []
    
    if not osm_data or 'elements' not in osm_data:
        return {
            "type": "FeatureCollection",
            "features": []
        }
    
    for element in osm_data['elements']:
        feature = {
            "type": "Feature",
            "properties": {},
            "geometry": None,
            "id": f"{element['type']}/{element['id']}"
        }
        
        # Thêm properties từ tags
        if 'tags' in element:
            feature['properties'] = element['tags'].copy()
        feature['properties']['@id'] = f"{element['type']}/{element['id']}"
        
        # Xử lý geometry theo loại element
        if element['type'] == 'node':
            if 'lat' in element and 'lon' in element:
                feature['geometry'] = {
                    "type": "Point",
                    "coordinates": [element['lon'], element['lat']]
                }
                
        elif element['type'] == 'way':
            # Nếu có center (từ out center), dùng center
            if 'center' in element:
                feature['geometry'] = {
                    "type": "Point",
                    "coordinates": [element['center']['lon'], element['center']['lat']]
                }
                feature['properties']['@geometry'] = 'center'
            # Nếu có geometry nodes
            elif 'geometry' in element:
                coords = [[n['lon'], n['lat']] for n in element['geometry']]
                if coords[0] == coords[-1] and len(coords) >= 4:
                    feature['geometry'] = {
                        "type": "Polygon",
                        "coordinates": [coords]
                    }
                else:
                    feature['geometry'] = {
                        "type": "LineString",
                        "coordinates": coords
                    }
                    
        elif element['type'] == 'relation':
            # Dùng center cho relation
            if 'center' in element:
                feature['geometry'] = {
                    "type": "Point",
                    "coordinates": [element['center']['lon'], element['center']['lat']]
                }
                feature['properties']['@geometry'] = 'center'
        
        # Chỉ thêm feature nếu có geometry hợp lệ
        if feature['geometry'] is not None:
            features.append(feature)
    
    return {
        "type": "FeatureCollection",
        "generator": "GreenMap-Data/data_collection.ipynb",
        "copyright": "The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.",
        "timestamp": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "features": features
    }


print("✅ Đã định nghĩa các hàm query thành công!")

---
## 3. 📋 Định nghĩa các Query

In [ ]:
# Bounding box cho khu vực Hà Nội
# Format: (min_lat, min_lon, max_lat, max_lon)
BBOX = "20.57, 105.28, 21.39, 106.02"

# Dictionary chứa tất cả các queries
QUERIES = {
    "bicycle_rental": {
        "name": "Điểm thuê xe đạp",
        "emoji": "🚲",
        "filename": "bicycle_rental.geojson",
        "timeout": 60,
        "query": f"""
[out:json][timeout:25];
(
  node["amenity"="bicycle_rental"]({BBOX});
  way["amenity"="bicycle_rental"]({BBOX});
  relation["amenity"="bicycle_rental"]({BBOX});
);
out body center;
>;
out skel qt;
"""
    },
    
    "charging_station": {
        "name": "Trạm sạc xe điện",
        "emoji": "⚡",
        "filename": "charging_station.geojson",
        "timeout": 60,
        "query": f"""
[out:json][timeout:25];
(
  node["amenity"="charging_station"]({BBOX});
  way["amenity"="charging_station"]({BBOX});
  relation["amenity"="charging_station"]({BBOX});
);
out body center;
>;
out skel qt;
"""
    },
    
    "parks": {
        "name": "Công viên",
        "emoji": "🌳",
        "filename": "park.geojson",
        "timeout": 120,
        "query": f"""
[out:json][timeout:60];
(
  node["leisure"="park"]({BBOX});
  way["leisure"="park"]({BBOX});
  relation["leisure"="park"]({BBOX});
);
out body center;
>;
out skel qt;
"""
    },
    
    "tourist_attractions": {
        "name": "Điểm du lịch",
        "emoji": "🏛️",
        "filename": "tourist_attractions.geojson",
        "timeout": 300,
        "query": f"""
[out:json][timeout:180];
(
  // Điểm du lịch chính
  node["tourism"]({BBOX});
  way["tourism"]({BBOX});
  relation["tourism"]({BBOX});
  
  // Di tích lịch sử
  node["historic"]({BBOX});
  way["historic"]({BBOX});
  relation["historic"]({BBOX});
  
  // Di sản văn hóa
  node["heritage"]({BBOX});
  way["heritage"]({BBOX});
  relation["heritage"]({BBOX});
);
out body center;
>;
out skel qt;
"""
    }
}

print("✅ Đã cấu hình các queries:")
for key, config in QUERIES.items():
    print(f"   {config['emoji']} {config['name']} → {config['filename']}")

---
## 4. 🔄 Tự động thu thập dữ liệu

In [ ]:
def collect_all_data(queries=QUERIES, data_dir=DATA_DIR, delay_between_requests=5):
    """
    Tự động thu thập tất cả dữ liệu từ Overpass API.
    
    Args:
        queries (dict): Dictionary chứa các query
        data_dir (str): Thư mục lưu dữ liệu
        delay_between_requests (int): Thời gian chờ giữa các request (giây)
    
    Returns:
        dict: Kết quả thu thập cho mỗi loại dữ liệu
    """
    # Tạo thư mục Data nếu chưa có
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
        print(f"📁 Đã tạo thư mục '{data_dir}'")
    
    results = {}
    total = len(queries)
    
    print("\n" + "="*60)
    print("🚀 BẮT ĐẦU THU THẬP DỮ LIỆU")
    print("="*60 + "\n")
    
    for i, (key, config) in enumerate(queries.items(), 1):
        print(f"\n[{i}/{total}] {config['emoji']} Đang thu thập: {config['name']}")
        print("-" * 40)
        
        # Gọi Overpass API
        osm_data = query_overpass(config['query'], timeout=config['timeout'])
        
        if osm_data:
            # Chuyển đổi sang GeoJSON
            geojson_data = osm_to_geojson(osm_data)
            feature_count = len(geojson_data['features'])
            
            # Lưu file
            filepath = os.path.join(data_dir, config['filename'])
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(geojson_data, f, ensure_ascii=False, indent=2)
            
            print(f"  💾 Đã lưu {feature_count} điểm vào '{filepath}'")
            
            results[key] = {
                'success': True,
                'count': feature_count,
                'filepath': filepath
            }
        else:
            print(f"  ⚠️ Không thu thập được dữ liệu cho {config['name']}")
            results[key] = {
                'success': False,
                'count': 0,
                'filepath': None
            }
        
        # Chờ trước request tiếp theo (trừ request cuối)
        if i < total:
            print(f"  ⏳ Đợi {delay_between_requests} giây trước request tiếp theo...")
            time.sleep(delay_between_requests)
    
    print("\n" + "="*60)
    print("✅ HOÀN TẤT THU THẬP DỮ LIỆU")
    print("="*60)
    
    return results

print("✅ Đã định nghĩa hàm thu thập dữ liệu!")
print("\n💡 Chạy cell tiếp theo để bắt đầu thu thập dữ liệu...")

In [ ]:
# ⚠️ Chạy cell này để BẮT ĐẦU THU THẬP DỮ LIỆU
# Quá trình có thể mất vài phút tùy thuộc vào tốc độ mạng

collection_results = collect_all_data(
    queries=QUERIES,
    data_dir=DATA_DIR,
    delay_between_requests=5  # Chờ 5 giây giữa các request
)

---
## 5. 📊 Hiển thị thống kê

In [ ]:
def show_statistics(data_dir=DATA_DIR):
    """
    Đọc các file GeoJSON và hiển thị thống kê.
    
    Args:
        data_dir (str): Thư mục chứa dữ liệu
    
    Returns:
        pd.DataFrame: Bảng thống kê
    """
    stats = []
    
    for key, config in QUERIES.items():
        filepath = os.path.join(data_dir, config['filename'])
        
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            feature_count = len(data.get('features', []))
            file_size = os.path.getsize(filepath) / 1024  # KB
            timestamp = data.get('timestamp', 'N/A')
            
            stats.append({
                'Loại dữ liệu': f"{config['emoji']} {config['name']}",
                'File': config['filename'],
                'Số lượng điểm': feature_count,
                'Kích thước (KB)': round(file_size, 2),
                'Cập nhật': timestamp
            })
        else:
            stats.append({
                'Loại dữ liệu': f"{config['emoji']} {config['name']}",
                'File': config['filename'],
                'Số lượng điểm': 0,
                'Kích thước (KB)': 0,
                'Cập nhật': 'Chưa có file'
            })
    
    df = pd.DataFrame(stats)
    return df


# Hiển thị thống kê
print("\n📊 THỐNG KÊ DỮ LIỆU")
print("="*80)

stats_df = show_statistics()
display(stats_df)

# Tổng kết
total_points = stats_df['Số lượng điểm'].sum()
total_size = stats_df['Kích thước (KB)'].sum()

print(f"\n📈 TỔNG KẾT:")
print(f"   • Tổng số điểm dữ liệu: {total_points:,}")
print(f"   • Tổng dung lượng: {total_size:.2f} KB ({total_size/1024:.2f} MB)")

---
## 6. 🗺️ Visualize trên bản đồ (Optional)

In [ ]:
def create_map(data_dir=DATA_DIR, max_points_per_layer=500):
    """
    Tạo bản đồ tương tác với tất cả các điểm dữ liệu.
    
    Args:
        data_dir (str): Thư mục chứa dữ liệu
        max_points_per_layer (int): Số điểm tối đa mỗi layer (để tránh lag)
    
    Returns:
        folium.Map: Đối tượng bản đồ folium
    """
    if not FOLIUM_AVAILABLE:
        print("❌ Folium không khả dụng. Chạy: pip install folium")
        return None
    
    # Tọa độ trung tâm Hà Nội
    hanoi_center = [21.0285, 105.8542]
    
    # Tạo bản đồ
    m = folium.Map(
        location=hanoi_center,
        zoom_start=11,
        tiles='OpenStreetMap'
    )
    
    # Màu sắc cho từng loại dữ liệu
    colors = {
        'bicycle_rental': 'blue',
        'charging_station': 'orange',
        'parks': 'green',
        'tourist_attractions': 'red'
    }
    
    icons = {
        'bicycle_rental': 'bicycle',
        'charging_station': 'bolt',
        'parks': 'tree',
        'tourist_attractions': 'camera'
    }
    
    for key, config in QUERIES.items():
        filepath = os.path.join(data_dir, config['filename'])
        
        if not os.path.exists(filepath):
            print(f"⚠️ Không tìm thấy file: {filepath}")
            continue
        
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        features = data.get('features', [])
        
        # Giới hạn số điểm để tránh lag
        if len(features) > max_points_per_layer:
            print(f"⚠️ {config['name']}: Chỉ hiển thị {max_points_per_layer}/{len(features)} điểm")
            features = features[:max_points_per_layer]
        
        # Tạo MarkerCluster cho mỗi loại
        cluster = MarkerCluster(name=f"{config['emoji']} {config['name']}")
        
        for feature in features:
            geom = feature.get('geometry')
            props = feature.get('properties', {})
            
            if not geom:
                continue
            
            # Lấy tọa độ (chỉ hỗ trợ Point)
            if geom['type'] == 'Point':
                coords = geom['coordinates']
                lat, lon = coords[1], coords[0]
            elif geom['type'] == 'Polygon':
                # Lấy centroid của polygon
                coords = geom['coordinates'][0]
                lat = sum(c[1] for c in coords) / len(coords)
                lon = sum(c[0] for c in coords) / len(coords)
            else:
                continue
            
            # Tạo popup content
            name = props.get('name', props.get('operator', 'Không có tên'))
            popup_content = f"<b>{name}</b><br>"
            popup_content += f"<i>Loại: {config['name']}</i>"
            
            # Thêm marker
            folium.Marker(
                location=[lat, lon],
                popup=folium.Popup(popup_content, max_width=200),
                icon=folium.Icon(
                    color=colors.get(key, 'gray'),
                    icon=icons.get(key, 'info-sign'),
                    prefix='fa'
                )
            ).add_to(cluster)
        
        cluster.add_to(m)
        print(f"✅ Đã thêm {len(features)} điểm {config['name']} vào bản đồ")
    
    # Thêm layer control
    folium.LayerControl().add_to(m)
    
    return m


print("✅ Đã định nghĩa hàm tạo bản đồ!")
print("\n💡 Chạy cell tiếp theo để hiển thị bản đồ...")

In [ ]:
# Tạo và hiển thị bản đồ
print("🗺️ ĐANG TẠO BẢN ĐỒ...")
print("="*60)

greenmap = create_map(data_dir=DATA_DIR, max_points_per_layer=500)

if greenmap:
    print("\n✅ Bản đồ đã sẵn sàng! Cuộn xuống để xem.")
    print("💡 Mẹo: Nhấp vào các marker để xem thông tin chi tiết.")
    print("💡 Sử dụng Layer Control (góc trên bên phải) để bật/tắt các lớp.")
    
    # Hiển thị bản đồ
    display(greenmap)
else:
    print("\n❌ Không thể tạo bản đồ. Vui lòng kiểm tra lại.")

---
## 🎉 Hoàn tất!

Bạn đã thu thập thành công dữ liệu từ OpenStreetMap. Dữ liệu được lưu trong thư mục `Data/`.

### 📁 Các file được tạo:
- `Data/bicycle_rental.geojson` - Điểm thuê xe đạp
- `Data/charging_station.geojson` - Trạm sạc xe điện
- `Data/park.geojson` - Công viên
- `Data/tourist_attractions.geojson` - Điểm du lịch

### 🔗 Tài liệu tham khảo:
- [OpenStreetMap](https://www.openstreetmap.org/)
- [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API)
- [Overpass Turbo](https://overpass-turbo.eu/)
- [SUMO Documentation](https://sumo.dlr.de/docs/)